# Social Behavior Processing Pipeline

**Project:** Social Single/Group Housing Behavior Analysis  
**Author:** Mir Qi 
**Last Updated:** October 20, 2024

## Overview
This notebook processes and validates multi-camera behavioral recordings for social interaction experiments. The pipeline includes:
1. Scanning and logging folder structures
2. Reading and consolidating parquet logs
3. Data quality checks and filtering
4. Visualization of COM (center of mass) trajectories
5. DANNCE 3D pose validation

---
## 1. Setup & Configuration

In [2]:
import sys
import os
from pathlib import Path

# Add project root to path
sys.path.append(os.path.abspath('../..'))

# Import custom modules
from status_fields.status_fields_config_oct3v1_brws_250525 import STATUS_FIELDS_CONFIG
from utlis.scan_engine_utlis.scan_eng_big_utlis import log_folder_to_parquet_sep
from utlis.scan_engine_utlis.scan_engine_utlis import read_all_parquet_files

In [3]:
# Configuration
BASE_FOLDER = "/data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused"
FAILED_PATHS_FILE = os.path.join(BASE_FOLDER, "bad_calib.txt")

# Rescan settings
FORCE_RESCAN_SESSIONS = []  # Add specific sessions to force rescan: [('2023-10-01', '001'), ...]
RESCAN_THRESHOLD_DAYS = 0.000000001  # Days threshold for automatic rescan

---
## 2. Scan Folders & Generate Logs

This step scans the data directory structure and creates/updates parquet log files for each session.

In [4]:
# Scan all folders and create logs
log_folder_to_parquet_sep(
    BASE_FOLDER,
    FAILED_PATHS_FILE,
    STATUS_FIELDS_CONFIG,
    force_rescan_rec_files=FORCE_RESCAN_SESSIONS,
    rescan_threshold_days=RESCAN_THRESHOLD_DAYS
)

print("\n✓ Folder scanning complete")

Log for 0single5_group5_ saved at /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_08_04/0single5_group5_/folder_log.parquet
Log for 0single5_group5 saved at /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_08_04/0single5_group5/folder_log.parquet
Log for 0single5_group5_3 saved at /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_08_04/0single5_group5_3/folder_log.parquet
Log for extrinsics saved at /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_10_03_17_00/extrinsics/folder_log.parquet
Log for 0single5_group2 saved at /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_10_03/0single5_group2/folder_log.parquet
Log for 0single3_group4 saved at /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_10_03/0single3_group4/folder_log.parquet
Log for 0single4_group3 saved at /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_10_03/0single4_group3/folder_log.parquet
Log for 0single2_group5 s

---
## 3. Load & Consolidate Data

Read all parquet files into a single dataframe for analysis.

In [6]:
# Load all parquet logs
all_df = read_all_parquet_files(BASE_FOLDER).to_pandas()

print(f"Total sessions loaded: {len(all_df)}")
print(f"Columns: {list(all_df.columns)}")

Total sessions loaded: 35
Columns: ['mir_generate_param', 'sync', 'mini_6cam_map', 'dropf_handle', 'com', 'com_vis', 'social', 'miniscope', 'test', 'after_oxytocin', 'before_oxytocin', 'dannce', 'dannce_vis', 'mini_rec_sync', 'mini_rec_sync_com', 'rec_file', 'scan_time', 'rec_path', 'date_folder', 'calib_files']


---
## 4. Data Overview & Quality Checks

In [7]:
# Display basic info
print("Dataset Shape:", all_df.shape)
print("\nFirst few rows:")
all_df.head()

Dataset Shape: (35, 20)

First few rows:


,mir_generate_param,sync,mini_6cam_map,dropf_handle,com,com_vis,social,miniscope,test,after_oxytocin,before_oxytocin,dannce,dannce_vis,mini_rec_sync,mini_rec_sync_com,rec_file,scan_time,rec_path,date_folder,calib_files
0,1,3,3,3,0,0,1,0,0,0,0,0,0,0,0,0single5_group2,2025-10-31T15:05:30.651867,/data/big_rim/rsync_dcc_sum/minji_social_singl...,2025_10_03,"[calib_after, calib_14_25_newintrinsics, calib..."
1,1,3,3,3,0,0,1,0,0,0,0,0,0,0,0,0single3_group4,2025-10-31T15:05:30.652320,/data/big_rim/rsync_dcc_sum/minji_social_singl...,2025_10_03,"[calib_after, calib_14_25_newintrinsics, calib..."
2,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0single4_group3,2025-10-31T15:05:30.652784,/data/big_rim/rsync_dcc_sum/minji_social_singl...,2025_10_03,"[calib_after, calib_14_25_newintrinsics, calib..."
3,1,3,3,3,0,0,1,0,0,0,0,0,0,0,0,0single2_group5,2025-10-31T15:05:30.653870,/data/big_rim/rsync_dcc_sum/minji_social_singl...,2025_10_03,"[calib_after, calib_14_25_newintrinsics, calib..."
4,1,3,3,3,0,0,1,0,0,0,0,0,0,0,0,0single5_group2_1,2025-10-31T15:05:30.654881,/data/big_rim/rsync_dcc_sum/minji_social_singl...,2025_10_03,"[calib_after, calib_14_25_newintrinsics, calib..."


In [9]:
# Summary statistics for processing status
print("\n=== Processing Status Summary ===")

status_columns = ['mir_generate_param', 'sync', 'mini_6cam_map', 'dropf_handle', 
                 'com', 'com_vis', 'social', 'dannce', 'dannce_vis']

for col in status_columns:
    if col in all_df.columns:
        completed = all_df[col].sum()
        total = len(all_df)
        pct = (completed / total * 100) if total > 0 else 0
        print(f"{col:20s}: {completed:3d}/{total:3d} ({pct:5.1f}%)")


=== Processing Status Summary ===


TypeError: unsupported operand type(s) for /: 'str' and 'int'

In [ ]:
# Check for incomplete sessions
print("\n=== Sessions Missing COM Processing ===")
missing_com = all_df[all_df['com'] == 0]
print(f"Found {len(missing_com)} sessions without COM processing")

if len(missing_com) > 0:
    print(missing_com[['date_folder', 'rec_file']].head(10))

---
## 5. Filter & Select Sessions

Filter the dataset based on specific criteria for further processing.

In [ ]:
# Example: Filter for social sessions with COM but missing COM visualization
filtered_table = all_df[
    (all_df['social'] == 1) & 
    (all_df['com'] == 1) & 
    (all_df['com_vis'] == 0)
]

print(f"Filtered sessions: {len(filtered_table)}")
print("\nSample of filtered sessions:")
filtered_table[['date_folder', 'rec_file', 'com', 'com_vis']].head(10)

---
## 6. Visualize Session Timeline

Create a timeline visualization showing when sessions were recorded.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from datetime import datetime

def visualize_sessions_timeline(df, base_folder, max_sessions=20):
    """
    Display COM circle images for sessions in chronological order.
    
    Args:
        df: Dataframe with 'date_folder' and 'rec_file' columns
        base_folder: Base path to session folders
        max_sessions: Maximum number of sessions to display
    """
    sessions = []
    
    # Collect session paths and timestamps
    for _, row in df.iterrows():
        session_path = os.path.join(base_folder, row['date_folder'], row['rec_file'])
        frametime_path = os.path.join(session_path, "videos", "Camera1", "frametimes.mat")
        
        if os.path.isfile(frametime_path):
            mtime = os.path.getmtime(frametime_path)
            sessions.append((session_path, mtime, row['rec_file']))
    
    # Sort by timestamp and limit
    sessions.sort(key=lambda x: x[1])
    sessions = sessions[:max_sessions]
    
    # Create visualization
    n = len(sessions)
    if n == 0:
        print("No sessions found for visualization")
        return
    
    fig, axes = plt.subplots(1, n, figsize=(min(n * 3, 60), 3))
    if n == 1:
        axes = [axes]
    
    for ax, (session_path, mtime, rec_name) in zip(axes, sessions):
        img_path = os.path.join(session_path, "COM/predict00/vis", "com_circle.png")
        
        if os.path.isfile(img_path):
            img = mpimg.imread(img_path)
            ax.imshow(img)
        else:
            ax.text(0.5, 0.5, "No Image", ha="center", va="center", 
                   bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))
        
        timestamp = datetime.fromtimestamp(mtime).strftime("%Y-%m-%d\n%H:%M")
        ax.set_title(f"{rec_name}\n{timestamp}", fontsize=8)
        ax.axis("off")
    
    plt.tight_layout()
    plt.suptitle("Session Timeline (Chronological)", fontsize=14, y=1.02)
    plt.show()

# Visualize filtered sessions
visualize_sessions_timeline(filtered_table, BASE_FOLDER, max_sessions=15)

---
## 7. Generate COM Visualizations

Process filtered sessions to generate center-of-mass trajectory visualizations for social experiments.

In [ ]:
from utlis.vis_valid_utlis.scom_traga_utlis import plot_com_all_social

# Convert filtered table to list of records
records = [
    {
        'date_folder': date_folder.as_py(),
        'rec_file': rec_file.as_py()
    }
    for date_folder, rec_file in zip(filtered_table['date_folder'], filtered_table['rec_file'])
]

print(f"Processing {len(records)} sessions...\n")

# Process each session
for i, record in enumerate(records, 1):
    session_path = os.path.join(BASE_FOLDER, record['date_folder'], record['rec_file'])
    print(f"[{i}/{len(records)}] {session_path}")
    
    try:
        plot_com_all_social(
            session_path,
            perform_generate_com_video=True
        )
        print("  ✓ Complete")
    except Exception as e:
        print(f"  ✗ Error: {e}")
        continue

print("\n=== COM visualization complete ===")

---
## 8. Validate DANNCE 3D Poses

Run validation on DANNCE 3D pose estimation results.

In [ ]:
from useful_files.sophie_check_dannce_mir_modif import dannce_valid
from concurrent.futures import ProcessPoolExecutor, as_completed

# Filter for sessions needing DANNCE validation
dannce_sessions = all_df[
    (all_df['dannce'] == 1) & 
    (all_df['dannce_vis'] == 0)
]

records = [
    {
        'date_folder': date_folder.as_py(),
        'rec_file': rec_file.as_py()
    }
    for date_folder, rec_file in zip(dannce_sessions['date_folder'], dannce_sessions['rec_file'])
]

print(f"Processing {len(records)} DANNCE sessions...\n")

def process_dannce_session(record):
    """Process a single DANNCE session with error handling."""
    session_path = os.path.join(BASE_FOLDER, record['date_folder'], record['rec_file'])
    try:
        dannce_valid(session_path)
        return f"✓ {session_path}"
    except Exception as e:
        return f"✗ {session_path}: {e}"

# Parallel processing
with ProcessPoolExecutor() as executor:
    futures = [executor.submit(process_dannce_session, record) for record in records]
    
    for future in as_completed(futures):
        result = future.result()
        print(result)

print("\n=== DANNCE validation complete ===")

---
## 9. Summary & Next Steps

In [ ]:
# Final summary
print("\n" + "="*50)
print("PROCESSING SUMMARY")
print("="*50)
print(f"Total sessions: {len(all_df)}")
print(f"Sessions processed: {len(filtered_table)}")
print(f"\nNext steps:")
print("  1. Review generated visualizations")
print("  2. Check for any failed sessions")
print("  3. Update processing flags in database")
print("="*50)

---
## Notes & Documentation

### Key Files
- **STATUS_FIELDS_CONFIG**: Defines processing pipeline stages
- **bad_calib.txt**: List of sessions with calibration issues

### Processing Stages
1. `mir_generate_param` - Mirror parameter generation
2. `sync` - Camera synchronization
3. `mini_6cam_map` - 6-camera mapping
4. `dropf_handle` - Dropped frame handling
5. `com` - Center of mass calculation
6. `com_vis` - COM visualization
7. `social` - Social interaction flags
8. `dannce` - 3D pose estimation
9. `dannce_vis` - DANNCE visualization

### Troubleshooting
- If sessions are missing, check file permissions
- For failed visualizations, verify image paths
- Parallel processing may require adjusting worker count based on system resources